## Entorno: Data, docker y spark

Se descarga la data y se configura el entorno con docker y spark

#### Installs

Instala Jupyter (`notebook`), la dependencia necesaria para poder abrir y correr este notebook

In [1]:
%pip install -q notebook


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


#### Imports

Importa las librerías estándar (subprocess, sys, pathlib) que usan las demás celdas del notebook

In [2]:
import subprocess, sys
from pathlib import Path

#### Utilidades

Define `sh()`, la función que ejecuta comandos de shell mostrando su salida en vivo, y calcula la ruta raíz del proyecto

In [3]:
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, cwd=proyecto, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        print(linea, end="")

    p.wait()

    if p.returncode != 0:
        raise RuntimeError(p.returncode)

proyecto = Path.cwd()
if not (proyecto / "docker-compose.yml").exists() and (proyecto.parent / "docker-compose.yml").exists():
    proyecto = proyecto.parent

### 1. Descargar data

- Ejecuta `data/download.py` para descargar los archivos JSON de contrataciones

- El script salta los meses que ya están en `data/raw` y solo baja los que faltan

In [4]:
sh(f'"{sys.executable}" data/download.py')
print(f"{len(list((proyecto / 'data' / 'raw').glob('*/*.json')))} archivos en data/raw")

36 archivos en data/raw


### 2. Verificación docker

Comprueba que el CLI de Docker y el plugin `compose` estén instalados en la máquina

In [5]:
sh("docker --version")
sh("docker compose version")

Docker version 29.7.2, build a7dcaa6


Docker Compose version v5.4.0


### 3. Iniciar cluster spark

- Levanta el cluster de Spark (master + 2 workers) y Jupyter

- Kernel picker → "Existing Jupyter Server..." → `http://localhost:8888`

In [6]:
sh("docker compose up -d")
sh("docker compose ps")

 Container dm-proj2-spark-master-1 Starting 


 Container dm-proj2-spark-master-1 Started 
 Container dm-proj2-spark-worker-2-1 Starting 
 Container dm-proj2-spark-worker-1 Starting 
 Container dm-proj2-spark-worker-1 Started 
 Container dm-proj2-jupyter-1 Starting 
 Container dm-proj2-spark-worker-2-1 Started 


 Container dm-proj2-jupyter-1 Started 
NAME                        IMAGE                                          COMMAND                  SERVICE          CREATED      STATUS                                     PORTS
dm-proj2-jupyter-1          quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   jupyter          8 days ago   Up Less than a second (health: starting)   0.0.0.0:4040->4040/tcp, [::]:4040->4040/tcp, 0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp
dm-proj2-spark-master-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-master     8 days ago   Up Less than a second (health: starting)   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp
dm-proj2-spark-worker-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-worker     8 days ago   Up Less than a second (health: starting)   0.0.0.0:8081->8081/tcp, [::]:8081->8081/tcp
dm-proj2-spark-worker-2-1   quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g --

### 4. Monitorear cluster

- Muestra el estado de los contenedores y el consumo de CPU/RAM del cluster de Spark

- UI de Spark: `http://localhost:8080` (master) y `http://localhost:4040` (job activo)

In [7]:
sh("docker compose ps")
sh("docker stats --no-stream")

NAME                        IMAGE                                          COMMAND                  SERVICE          CREATED      STATUS                                     PORTS
dm-proj2-jupyter-1          quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   jupyter          8 days ago   Up Less than a second (health: starting)   0.0.0.0:4040->4040/tcp, [::]:4040->4040/tcp, 0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp
dm-proj2-spark-master-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-master     8 days ago   Up Less than a second (health: starting)   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp
dm-proj2-spark-worker-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-worker     8 days ago   Up Less than a second (health: starting)   0.0.0.0:8081->8081/tcp, [::]:8081->8081/tcp
dm-proj2-spark-worker-2-1   quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-worker-2   8 days a

CONTAINER ID   NAME                        CPU %     MEM USAGE / LIMIT     MEM %     NET I/O         BLOCK I/O         PIDS
b3aa4c539144   dm-proj2-jupyter-1          24.02%    64.84MiB / 7.651GiB   0.83%     1.64kB / 126B   41.9MB / 4.1kB    3
88d9ee148377   dm-proj2-spark-worker-2-1   30.12%    74.14MiB / 7.651GiB   0.95%     1.78kB / 126B   23.2MB / 36.9kB   21
5d5c33f7e7ba   dm-proj2-spark-worker-1     31.92%    89.22MiB / 7.651GiB   1.14%     1.69kB / 126B   41.1MB / 36.9kB   21
c289e973ffd4   dm-proj2-spark-master-1     9.96%     56.1MiB / 7.651GiB    0.72%     1.82kB / 84B    38.5MB / 4.1kB    23


### 5. Apagar cluster

Detiene y borra los contenedores, pero la data y los resultados quedan en disco

In [8]:
# sh("docker compose down")